# Chapter 2: Simulation & Data

This notebook walks through the SO-100 pick-and-place simulator, a scripted
baseline policy, and the LeRobot SO-101 expert dataset that feeds the
normalized DataLoader Chapter 3 imports. The arm family is SO-100 in sim,
SO-101 on hardware; the small kinematic gap is a Chapter 9 topic. Each
section maps cell-by-cell to a section of the book chapter; listings are
reproduced verbatim from the chapter plan.

**Run locally:** `pip install -e ".[dev,data,sim]"` from the repo root, then
open this notebook in Jupyter.

**Run on Colab:** the cell below detects Colab and installs the Vulkan ICDs
ManiSkill needs before doing the pip install.

In [ ]:
# Colab-only setup: install Vulkan ICDs + the chapter package.
# On a local machine with the package already installed, this cell is a no-op.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !mkdir -p /usr/share/vulkan/icd.d
    !wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
    !wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json
    !mv nvidia_icd.json /usr/share/vulkan/icd.d
    !mv 10_nvidia.json /usr/share/glvnd/egl_vendor.d/10_nvidia.json
    !apt-get install -y --no-install-recommends libvulkan-dev
    !pip install -q "lrm-ch02[data,sim] @ git+https://github.com/Large-Robotics-Models-From-Scratch/lrm-code-chapter-2.git@main"


## 2.1 The SO-100 Pick-and-Place Environment

We use ManiSkill3's `PickCubeSO100-v1` task. The arm is SO-100 (close enough
to SO-101 that the kinematic gap is a Chapter 9 topic), the cube spawns in a
fixed region, and the goal is to drop the cube inside a target zone.

### Listing 2.1: Installing the SO-100 sim and creating the environment

Constructs the env with image observations and the default joint-space
control mode. The action space is `Box(6,)` — one delta per SO-100 joint.

In [ ]:
# pip install lerobot mani-skill
import gymnasium as gym
import mani_skill.envs

env = gym.make(
    "PickCubeSO100-v1",
    obs_mode="rgb",
    control_mode="pd_joint_delta_pos",
    render_mode="rgb_array",
)
obs, info = env.reset(seed=42)
print(f"Observation keys: {list(obs.keys())}")
print(f"Action space: {env.action_space}")


### Listing 2.2: Running a random agent on PickCubeSO100-v1

`run_random_agent` samples uniformly from the action space across `n_episodes`
and returns `(success_rate, mean_return)`. Random actions almost never solve
pick-and-place — this is the performance floor every learned policy must
clear.

In [ ]:
import numpy as np

def run_random_agent(env, n_episodes=10):
    successes, returns = 0, []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        ep_return = 0.0
        done = False
        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            ep_return += reward
            done = terminated or truncated
        successes += int(info.get("success", False))
        returns.append(ep_return)
    return successes / n_episodes, np.mean(returns)

success_rate, mean_return = run_random_agent(env)
print(f"Random agent: success={success_rate:.0%} "
      f"return={mean_return:.2f}")


## 2.2 A Scripted Policy

Seven phases — approach, descend, grasp, lift, transport, place, release —
expressed as keyframe poses. ManiSkill's `SO100ArmMotionPlanningSolver`
solves IK and steps the joint trajectory for each keyframe.

### Listing 2.3: A multi-phase scripted pick-and-place policy

In [ ]:
import numpy as np
import sapien

def run_scripted_episode(
        planner, grasp_pose, goal_pos):
    """Seven phases via the motion planner.

    approach → descend → grasp → lift → transport → place → release.
    """
    quat = grasp_pose.q
    goal = np.asarray(goal_pos)

    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.10]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.03]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.01]) * grasp_pose)
    planner.close_gripper(gripper_state=-0.8)
    planner.move_to_pose_with_screw(
        sapien.Pose([0, 0, 0.15]) * grasp_pose)
    planner.move_to_pose_with_screw(
        sapien.Pose(goal + np.array([0, 0, 0.15]), quat))
    planner.move_to_pose_with_screw(
        sapien.Pose(goal + np.array([0, 0, 0.02]), quat))
    planner.open_gripper()


### Listing 2.4: Evaluating the scripted policy

`run_scripted_agent` (from `ch02.scripted`) handles motion-planner setup,
grasp-pose computation, and success counting. We pass a state-mode env
since the policy reads cube/goal positions directly from `env.unwrapped`.

In [ ]:
from ch02.scripted import run_scripted_agent
from ch02.env import make_env

env = make_env(obs_mode="state", render_mode=None)
rate = run_scripted_agent(env, n_episodes=10)
print(f"Scripted agent success rate: {rate:.0%}")


## 2.3 The LeRobot Dataset Standard

The chapter's expert data comes from `lerobot/svla_so101_pickplace` —
50 episodes of real-hardware teleoperated pick-and-place at 30 fps. Each
frame has a 6-DOF joint state, two USB-camera streams (`up` and `side`),
the recorded teleoperation action, and per-episode metadata.

### Listing 2.5: Loading the SO-101 pick-and-place expert dataset

In [ ]:
from lerobot.datasets import LeRobotDataset

dataset = LeRobotDataset(
    "lerobot/svla_so101_pickplace",
)
print(f"Total frames: {len(dataset)}")
print(f"Episodes: {dataset.num_episodes}")
print(f"Features: {list(dataset.features.keys())}")


### Listing 2.6: Inspecting a single frame and one episode

Inspect frame 0's tensor shapes and dtypes, then collect every frame that
belongs to episode 0 to confirm episode-length variation across the 50
trajectories.

In [ ]:
frame = dataset[0]
for key, val in frame.items():
    if hasattr(val, "shape"):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
    else:
        print(f"  {key}: {val}")

ep_indices = [i for i in range(len(dataset))
              if dataset[i]["episode_index"] == 0]
print(f"\nEpisode 0 length: {len(ep_indices)} steps")
